# Lab 08 — dots.ocr: OCR con Vision-Language Model

> **GraphoLab** | Forensic Graphology Laboratory

**Modello:** `rednote-hilab/dots.ocr` (Hugging Face)  
**Task:** Trascrizione di testo manoscritto e stampato da immagini di documenti  
**Caso d'uso forense:** Testamenti, lettere anonime, documenti storici in italiano

---

## Come funziona dots.ocr

dots.ocr è fondamentalmente diverso da EasyOCR e TrOCR. Invece di una pipeline CNN+CRNN,
usa un **Vision-Language Model (VLM)** da **1.7 miliardi di parametri**:

```
EasyOCR / TrOCR:                    dots.ocr:
────────────────                    ─────────────────────────────
Immagine                            Immagine
   ↓                                   ↓
CRAFT (detector CNN)                Vision Encoder (ViT)
   ↓                                   ↓
CRNN (recognizer)                   Visual Tokens
   ↓                                   ↓
Testo                               LLM (1.7B params) ← comprende il contesto!
                                       ↓
                                    Testo
```

Il vantaggio chiave: il **componente LLM usa il contesto linguistico** per correggere
ambiguità visive. Per l'italiano, questo significa meno errori su parole con accenti,
apostrofi e congiunzioni (es. `è`, `l'arte`, `nell'atto`).

| Caratteristica | EasyOCR | TrOCR | **dots.ocr** |
|---|---|---|---|
| Architettura | CNN + CRNN | ViT + RoBERTa | ViT + LLM 1.7B |
| Comprensione layout | parziale | no | **si** (tabelle, formule) |
| Contesto linguistico | no | limitato (inglese) | **si** (100+ lingue) |
| Dimensione modello | ~100 MB | ~1.3 GB | ~3.5 GB (bf16) |
| Velocità su CPU | veloce | lenta | **molto lenta** |
| Qualita' su corsivo | media | media | **migliore** |

> **Paper:** [arxiv 2512.02498](https://arxiv.org/abs/2512.02498) — RedNote / Xiaohongshu, dic 2024

## 1. Verifica Hardware

dots.ocr e' pesante. Prima di caricare il modello controlliamo le risorse disponibili
e scegliamo la configurazione piu' adatta al laptop.

In [ ]:
import torch
import psutil
import platform

ram_gb   = psutil.virtual_memory().total / 1e9
ram_free = psutil.virtual_memory().available / 1e9
has_gpu  = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if has_gpu else 'N/A'
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9 if has_gpu else 0

print(f"Sistema       : {platform.system()} {platform.release()}")
print(f"CPU           : {platform.processor()[:60]}")
print(f"RAM totale    : {ram_gb:.1f} GB  (libera: {ram_free:.1f} GB)")
print(f"GPU           : {gpu_name}")
print(f"VRAM GPU      : {vram_gb:.1f} GB" if has_gpu else "VRAM GPU      : N/A")
print()

# Raccomandazione
if has_gpu and vram_gb >= 8:
    DEVICE = 'cuda'
    DTYPE  = torch.bfloat16
    ATTN   = 'flash_attention_2'
    print("[OK] GPU con VRAM >= 8 GB — usero' CUDA + bf16 + flash_attention_2 (configurazione ottimale)")
elif has_gpu and vram_gb >= 4:
    DEVICE = 'cuda'
    DTYPE  = torch.float16
    ATTN   = 'sdpa'
    print("[OK] GPU con VRAM 4-8 GB — usero' CUDA + fp16 + sdpa")
elif ram_free >= 8:
    DEVICE = 'cpu'
    DTYPE  = torch.float32
    ATTN   = 'eager'
    print("[OK] Solo CPU con RAM libera >= 8 GB — usero' CPU + fp32 (lento ma funziona)")
    print("     Stima tempo per immagine: 2-5 minuti su CPU moderna")
else:
    DEVICE = 'cpu'
    DTYPE  = torch.float32
    ATTN   = 'eager'
    print("[ATTENZIONE] RAM libera < 8 GB — il modello potrebbe non caricarsi completamente.")
    print("             Chiudi altre applicazioni prima di procedere.")

print(f"\nConfigurazione scelta: device={DEVICE}, dtype={DTYPE}, attn={ATTN}")

## 2. Installazione

dots.ocr non e' su PyPI. Richiede il clone del repo e l'installazione locale.
Eseguire **una volta sola** — la cella e' commentata per evitare reinstallazioni accidentali.

In [ ]:
# Decommenta ed esegui SOLO la prima volta
# -------------------------------------------------------
# import subprocess, sys
#
# # 1. Dipendenze base
# subprocess.run([sys.executable, '-m', 'pip', 'install',
#                 'transformers>=4.49', 'qwen_vl_utils',
#                 'accelerate', 'Pillow', 'psutil'], check=True)
#
# # 2. Clona il repo di dots.ocr (usa il nome 'DotsOCR' senza punti!)
# subprocess.run(['git', 'clone',
#                 'https://github.com/rednote-hilab/dots.ocr.git',
#                 'DotsOCR'], check=True)
#
# # 3. Installa il pacchetto locale
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', 'DotsOCR'], check=True)
#
# print('Installazione completata!')
# -------------------------------------------------------
print('Cella di installazione — decommenta per eseguire.')

## 3. Import e Utility

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import time

import torch
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Percorso root del progetto (notebook si trova in notebooks/)
ROOT = Path('..').resolve()
print(f'Root progetto: {ROOT}')
print(f'PyTorch: {torch.__version__}')

## 4. Caricamento del Modello

Il modello viene scaricato da Hugging Face la prima volta (~3.5 GB in bf16, ~7 GB in fp32)
e messo in cache in `~/.cache/huggingface/hub`.

> Su CPU la prima inferenza richiede 2-5 minuti. Le successive sono piu' veloci
> perche' il modello resta in RAM.

In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor

MODEL_ID = 'rednote-hilab/dots.ocr'

print(f'Caricamento {MODEL_ID} ...')
print(f'Device: {DEVICE} | dtype: {DTYPE} | attn: {ATTN}')
print('(Prima volta: scarica ~3.5 GB. Attendi.)')

t0 = time.time()

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

load_kwargs = dict(
    torch_dtype=DTYPE,
    trust_remote_code=True,
)
if DEVICE == 'cuda':
    load_kwargs['device_map'] = 'auto'
    if ATTN == 'flash_attention_2':
        load_kwargs['attn_implementation'] = 'flash_attention_2'
else:
    load_kwargs['device_map'] = 'cpu'

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()

print(f'Modello pronto in {time.time()-t0:.1f}s')

## 5. Funzione di Trascrizione

dots.ocr accetta messaggi nel formato chat (come ChatGPT): un'immagine + un prompt testuale
che specifica cosa estrarre. Le modalita' principali sono:

- `full_ocr` — trascrive tutto il testo mantenendo l'ordine di lettura
- `layout_parse` — restituisce anche la struttura (titoli, paragrafi, tabelle)
- `formula` — rileva formule matematiche

Per i nostri documenti forensi usiamo `full_ocr`.

In [ ]:
try:
    from dots_ocr.utils import dict_promptmode_to_prompt
    PROMPT_TEXT = dict_promptmode_to_prompt.get('full_ocr',
        'Please perform OCR on this image and output all text you can read, preserving line breaks.')
    print(f'Prompt ufficiale caricato: {PROMPT_TEXT[:80]}...')
except ImportError:
    # Fallback se dots_ocr non e' installato come pacchetto
    PROMPT_TEXT = (
        'Please perform OCR on this image. '
        'Output only the transcribed text, preserving line breaks and reading order.'
    )
    print('dots_ocr non trovato come pacchetto — uso prompt generico.')


def transcribe(image_path: str | Path, max_new_tokens: int = 1024) -> tuple[str, float]:
    """Trascrive il testo in un'immagine con dots.ocr.
    
    Args:
        image_path: percorso all'immagine
        max_new_tokens: token massimi generati (aumentare per documenti lunghi)
    
    Returns:
        (testo_trascritto, secondi_impiegati)
    """
    # Prepara il messaggio nel formato chat
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': str(image_path)},
                {'type': 'text',  'text': PROMPT_TEXT},
            ]
        }
    ]

    # Tokenizzazione
    try:
        from qwen_vl_utils import process_vision_info
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors='pt',
        )
    except ImportError:
        # Fallback senza qwen_vl_utils
        img = Image.open(image_path).convert('RGB')
        inputs = processor(
            text=PROMPT_TEXT,
            images=img,
            return_tensors='pt'
        )

    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    t0 = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    elapsed = time.time() - t0

    # Decodifica (rimuove i token di input dal risultato)
    generated = output_ids[:, inputs['input_ids'].shape[1]:]
    text_out  = processor.batch_decode(generated, skip_special_tokens=True)[0]
    return text_out.strip(), elapsed


def show_result(image_path: str | Path, text: str, elapsed: float) -> None:
    """Visualizza immagine e trascrizione affiancate."""
    img = Image.open(image_path)
    fig = plt.figure(figsize=(16, max(5, img.height / img.width * 8)))
    gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 1])

    ax_img  = fig.add_subplot(gs[0])
    ax_text = fig.add_subplot(gs[1])

    ax_img.imshow(img)
    ax_img.set_title('Immagine originale', fontsize=12)
    ax_img.axis('off')

    ax_text.text(
        0.03, 0.97, text,
        fontsize=10, va='top', wrap=True,
        fontfamily='monospace',
        transform=ax_text.transAxes,
        bbox=dict(boxstyle='round', facecolor='#f5f5dc', alpha=0.9)
    )
    ax_text.set_title(f'Trascrizione dots.ocr  ({elapsed:.1f}s)', fontsize=12)
    ax_text.axis('off')

    plt.tight_layout()
    plt.show()

print('Funzioni pronte.')

## Demo 1 — Campione writer_00 (testo manoscritto)

Trascriviamo uno dei campioni di writer_00 che usiamo anche per l'identificazione scrittore.
Ogni immagine e' 320x140 px e contiene 3 righe di testo italiano.

In [ ]:
sample_path = ROOT / 'data/samples/writer_00/sample_000.png'

print(f'Immagine: {sample_path}')
print('Avvio trascrizione ... (su CPU: 2-5 minuti)')

text, elapsed = transcribe(sample_path)

print(f'\nTrascrizione ({elapsed:.1f}s):')
print('-' * 40)
print(text)

show_result(sample_path, text, elapsed)

## Demo 2 — Documento testamento (documento completo)

Trascriviamo `testamento_writer00.png`, il documento fittizio composto da campioni
reali di writer_00. Questo e' il caso d'uso forense principale.

> Con `max_new_tokens=2048` diamo al modello spazio sufficiente per trascrivere
> un intero documento di piu' pagine.

In [ ]:
doc_path = ROOT / 'data/samples/testamento_writer00.png'

if not doc_path.exists():
    print(f'File non trovato: {doc_path}')
    print('Esegui prima scripts/create_testamento_writer00.py')
else:
    print(f'Immagine: {doc_path}')
    print('Avvio trascrizione documento completo ... (piu\' lungo del campione singolo)')

    text_doc, elapsed_doc = transcribe(doc_path, max_new_tokens=2048)

    print(f'\nTrascrizione ({elapsed_doc:.1f}s):')
    print('-' * 40)
    print(text_doc)

    show_result(doc_path, text_doc, elapsed_doc)

## Demo 3 — Immagine Lorella (scrittura reale del mondo reale)

Trascriviamo una delle immagini dal dataset Lorella — scrittura reale,
non campioni di dettato standardizzati.

In [ ]:
lorella_dir = ROOT / 'data/lorella'
lorella_images = sorted(lorella_dir.glob('*.png'))[:2]  # prime 2 per velocita'

if not lorella_images:
    print(f'Nessuna immagine trovata in {lorella_dir}')
else:
    for img_path in lorella_images:
        print(f'\n--- {img_path.name} ---')
        text_l, elapsed_l = transcribe(img_path)
        print(f'Trascrizione ({elapsed_l:.1f}s):')
        print(text_l)
        show_result(img_path, text_l, elapsed_l)

## Demo 4 — Confronto EasyOCR vs dots.ocr

Confronto diretto sullo stesso campione per valutare la differenza di qualita'.

In [ ]:
import numpy as np

compare_path = ROOT / 'data/samples/writer_00/sample_000.png'
img_np = np.array(Image.open(compare_path).convert('RGB'))

# --- EasyOCR ---
print('EasyOCR ...')
import easyocr
reader = easyocr.Reader(['it', 'en'], gpu=DEVICE == 'cuda')
t_easy = time.time()
easy_result = reader.readtext(img_np, detail=0, paragraph=True)
easy_text   = '\n'.join(easy_result)
easy_time   = time.time() - t_easy

# --- dots.ocr ---
print('dots.ocr ...')
dots_text, dots_time = transcribe(compare_path)

# --- Visualizzazione ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(img_np)
axes[0].set_title('Immagine originale', fontsize=12)
axes[0].axis('off')

for ax, title, text, t in [
    (axes[1], f'EasyOCR ({easy_time:.1f}s)',  easy_text, easy_time),
    (axes[2], f'dots.ocr ({dots_time:.1f}s)', dots_text, dots_time),
]:
    ax.text(0.05, 0.5, text or '(nessun risultato)',
            fontsize=12, va='center', fontfamily='monospace',
            transform=ax.transAxes,
            bbox=dict(boxstyle='round', facecolor='#f0f8ff', alpha=0.9))
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('Confronto EasyOCR vs dots.ocr', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nEasyOCR : {easy_text}')
print(f'\ndots.ocr: {dots_text}')

## Misurazione CER (Character Error Rate)

Se conosci il testo esatto dell'immagine, puoi misurare l'errore con il CER
(frazione di caratteri errati, 0 = perfetto, 1 = tutto sbagliato).

In [ ]:
def cer(reference: str, hypothesis: str) -> float:
    """Character Error Rate tramite distanza di edit."""
    r, h = list(reference.replace(' ', '')), list(hypothesis.replace(' ', ''))
    d = [[0] * (len(h) + 1) for _ in range(len(r) + 1)]
    for i in range(len(r) + 1): d[i][0] = i
    for j in range(len(h) + 1): d[0][j] = j
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            cost = 0 if r[i-1] == h[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)
    return d[len(r)][len(h)] / max(len(r), 1)


# Testo atteso per sample_000.png
# (da leggere manualmente dall'immagine)
ground_truth = "il gatto dorme sul tetto\nla casa e piccola e bella\noggi il cielo e molto blu"

cer_easy = cer(ground_truth, easy_text)
cer_dots = cer(ground_truth, dots_text)

print(f'Ground truth : {ground_truth!r}')
print(f'EasyOCR      : {easy_text!r}  →  CER = {cer_easy:.3f} ({cer_easy*100:.1f}%)')
print(f'dots.ocr     : {dots_text!r}  →  CER = {cer_dots:.3f} ({cer_dots*100:.1f}%)')

winner = 'dots.ocr' if cer_dots < cer_easy else 'EasyOCR'
print(f'\nModello migliore su questo campione: {winner}')

## Note Forensi

- **dots.ocr e' un VLM**: genera testo token per token. In rari casi puo' "allucinare"
  parole plausibili ma non presenti nell'immagine. Verificare sempre contro l'originale.

- **Velocita' su CPU**: 2-5 minuti per immagine su laptop moderno senza GPU. Accettabile
  per analisi forensi manuali, non adatto a pipeline automatizzate in tempo reale.

- **Qualita' su corsivo**: migliore di EasyOCR grazie al contesto linguistico LLM,
  ma non perfetto — la scrittura corsiva personale rimane la sfida principale.

- **Alternativa commerciale per qualita' massima**: [Transkribus](https://www.transkribus.org)
  ha modelli specializzati su manoscritti storici italiani.

- **Integrazione nella demo Gradio**: il modello e' troppo lento per una demo interattiva
  su laptop. Manteniamo EasyOCR nel tab HTR e usiamo dots.ocr solo offline (questo notebook).

---

**Lab precedente →** [07 — Named Entity Recognition](07_named_entity_recognition.ipynb)